# Food Review Pipeline (Wongnai)

Notebook รันทั้ง pipeline บนข้อมูล Wongnai (`review_body` / `stars`) ผ่านสคริปต์ใน `scripts/`

**ต้องรันจาก root ของ repo** (`foods-review-classification/`)

**ลำดับที่แนะนำ:** รัน **Section 1 Setup** ก่อน (สร้างตัวแปร `ROOT`) แล้วค่อยรัน cell อื่น — หรือรัน **Section 3 Paths** ซึ่งจะสร้าง `ROOT` ให้อัตโนมัติถ้ายังไม่มี  

**ข้อผิดพลาดที่พบบ่อย — อ่านก่อนรัน cell ถัดไป:** [docs/ERRORS_AND_FIXES.md](docs/ERRORS_AND_FIXES.md) (และ `COMMON ERRORS` ที่หัวไฟล์ใน `scripts/`, `src/rris/`)

| อาการ | สาเหตุ | แก้ใน notebook |
|--------|--------|----------------|
| `NameError: ROOT` | ข้าม Setup/Paths | รัน Section 3 |
| Kernel crashed | Aspect + โหลด ST ซ้ำ / VRAM เต็ม | `SKIP_ASPECTS_ON_SCORE = True`, `TRAIN_DEVICE = "cpu"` |
| Train ช้า/ค้าง | TF-IDF 79k แถว | ลด `DEMO_MAX_ROWS` |
| Eval metrics แปลก | เทรน 5k แต่ eval ทั้งไฟล์ | preprocess/train ชุดเดียวกัน |
| `bad_malloc` ตอน train | C: เต็ม / RAM | ตั้ง `SCRATCH_DIR` (§3) + `TRAIN_DEVICE=cpu`, `--n_jobs 1` |

## 1. Setup

สร้าง venv แล้วเปิด kernel จาก venv นั้น (แนะนำ):

```powershell
python -m venv .venv
.\.venv\Scripts\Activate.ps1
pip install -r requirements.txt
pip install -e .
```

ถ้าไม่ `pip install -e .` สคริปต์ยังรันได้เพราะมี `scripts/_bootstrap.py`

In [1]:
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "scripts" / "preprocess.py").exists():
    raise SystemExit(
        "Run this notebook from the repo root (foods-review-classification/). "
        f"Current cwd: {ROOT}"
    )

def run_script(script: str, *args: str) -> None:
    cmd = [sys.executable, str(ROOT / "scripts" / script), *args]
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=ROOT, check=True)

print("Repo root:", ROOT)

Repo root: f:\ComSci\Coding\Project\foods-review-classification


In [2]:
# Optional: install deps (skip if already done in your venv)
INSTALL_DEPS = False

if INSTALL_DEPS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=ROOT, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], cwd=ROOT, check=True)

## 2. Optional: check CUDA

In [3]:
try:
    import torch

    print("torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    print("PyTorch not installed — baseline XGBoost will use CPU (--device auto).")

torch: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 2060


## 3. Paths (Wongnai)

In [4]:
# Bootstrap ROOT / run_script if you skipped Section 1 (e.g. ran CUDA cell first)
from pathlib import Path
import subprocess
import sys

if "ROOT" not in globals():
    ROOT = Path.cwd()
    if not (ROOT / "scripts" / "preprocess.py").exists():
        raise SystemExit(
            "Run this notebook from the repo root (foods-review-classification/). "
            f"Current cwd: {ROOT}"
        )
    print("Auto-set ROOT from cwd:", ROOT)

if "run_script" not in globals():
    def run_script(script: str, *args: str) -> None:
        cmd = [sys.executable, str(ROOT / "scripts" / script), *args]
        print("$", " ".join(cmd))
        subprocess.run(cmd, cwd=ROOT, check=True)

RAW_CSV = ROOT / "data" / "wongnai-restaurant-review_train.csv"
PROCESSED = ROOT / "data" / "wongnai_processed.parquet"
ARTIFACT_DIR = ROOT / "artifacts" / "notebook_baseline"
METRICS_JSON = ROOT / "reports" / "notebook_baseline_metrics.json"
SAMPLE_PARQUET = ROOT / "data" / "wongnai_sample_500.parquet"
SCORED_OUT = ROOT / "outputs" / "notebook_scored_sample.parquet"

# Set to an int for a quick demo; None = full dataset (~79k rows, slower)
DEMO_MAX_ROWS: int | None = 5000
SCORE_SAMPLE_ROWS = 500

# Notebook-safe training: torch in kernel already uses GPU; XGB "auto" can OOM VRAM
TRAIN_DEVICE = "cpu"
NOTEBOOK_N_JOBS = 4
SKIP_ASPECTS_ON_SCORE = True

# Temp / HF cache on a drive with free space (None = auto; or r"D:\rris-scratch")
SCRATCH_DIR: str | None = str(ROOT / ".cache" / "tmp")


def scratch_args() -> list[str]:
    if SCRATCH_DIR:
        return ["--scratch_dir", str(SCRATCH_DIR)]
    return []


if not RAW_CSV.exists():
    raise FileNotFoundError(f"Missing Wongnai CSV: {RAW_CSV}")

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_JSON.parent.mkdir(parents=True, exist_ok=True)
SCORED_OUT.parent.mkdir(parents=True, exist_ok=True)

print("Input:", RAW_CSV)
print("Processed:", PROCESSED)
print("Artifacts:", ARTIFACT_DIR)
print("Demo max_rows:", DEMO_MAX_ROWS)

Input: f:\ComSci\Coding\Project\foods-review-classification\data\wongnai-restaurant-review_train.csv
Processed: f:\ComSci\Coding\Project\foods-review-classification\data\wongnai_processed.parquet
Artifacts: f:\ComSci\Coding\Project\foods-review-classification\artifacts\notebook_baseline
Demo max_rows: 5000


## 4. Preprocess → parquet

In [5]:
run_script(
    "preprocess.py",
    "--input", str(RAW_CSV),
    "--out", str(PROCESSED),
    *scratch_args(),
)

$ c:\Users\teera\AppData\Local\Programs\Python\Python310\python.exe f:\ComSci\Coding\Project\foods-review-classification\scripts\preprocess.py --input f:\ComSci\Coding\Project\foods-review-classification\data\wongnai-restaurant-review_train.csv --out f:\ComSci\Coding\Project\foods-review-classification\data\wongnai_processed.parquet


## 5. Train baseline (TF-IDF + XGBoost)

`--device auto` ใช้ GPU ถ้ามี CUDA; ถ้า VRAM เต็มให้เปลี่ยนเป็น `--device cpu`

In [6]:
train_args = [
    "--input", str(PROCESSED),
    "--out_dir", str(ARTIFACT_DIR),
    "--device", TRAIN_DEVICE,
    "--n_jobs", str(NOTEBOOK_N_JOBS),
]
if DEMO_MAX_ROWS is not None:
    train_args.extend(["--max_rows", str(DEMO_MAX_ROWS)])

run_script("train_baseline_xgb.py", *train_args, *scratch_args())

$ c:\Users\teera\AppData\Local\Programs\Python\Python310\python.exe f:\ComSci\Coding\Project\foods-review-classification\scripts\train_baseline_xgb.py --input f:\ComSci\Coding\Project\foods-review-classification\data\wongnai_processed.parquet --out_dir f:\ComSci\Coding\Project\foods-review-classification\artifacts\notebook_baseline --device cpu --n_jobs 4 --max_rows 5000


## 6. Evaluate

In [ ]:
run_script(
    "evaluate.py",
    "--input", str(PROCESSED),
    "--model_type", "baseline_xgb",
    "--artifact_dir", str(ARTIFACT_DIR),
    "--out", str(METRICS_JSON),
    "--n_jobs", str(NOTEBOOK_N_JOBS),  # -1 on ~19k rows can crash kernel (tokenize RAM spike)
    *scratch_args(),
)

import json

print(json.dumps(json.loads(METRICS_JSON.read_text(encoding="utf-8")), indent=2, ensure_ascii=False))

$ c:\Users\teera\AppData\Local\Programs\Python\Python310\python.exe f:\ComSci\Coding\Project\foods-review-classification\scripts\evaluate.py --input f:\ComSci\Coding\Project\foods-review-classification\data\wongnai_processed.parquet --model_type baseline_xgb --artifact_dir f:\ComSci\Coding\Project\foods-review-classification\artifacts\notebook_baseline --out f:\ComSci\Coding\Project\foods-review-classification\reports\notebook_baseline_metrics.json --n_jobs -1
{
  "accuracy": 0.5902456573131873,
  "classification_report": {
    "0": {
      "precision": 0.9454545454545454,
      "recall": 0.21224489795918366,
      "f1-score": 0.3466666666666667,
      "support": 245.0
    },
    "1": {
      "precision": 0.8388888888888889,
      "recall": 0.25900514579759865,
      "f1-score": 0.39580602883355176,
      "support": 583.0
    },
    "2": {
      "precision": 0.6350293542074364,
      "recall": 0.4527906976744186,
      "f1-score": 0.5286451262557698,
      "support": 4300.0
    },
    

: 

## 7. Score + flag (small sample)

In [ ]:
import pandas as pd

sample_df = pd.read_parquet(PROCESSED).head(SCORE_SAMPLE_ROWS)
sample_df.to_parquet(SAMPLE_PARQUET, index=False)
print(f"Wrote {len(sample_df)} rows -> {SAMPLE_PARQUET}")

score_args = [
    "--input", str(SAMPLE_PARQUET),
    "--model_type", "baseline_xgb",
    "--artifact_dir", str(ARTIFACT_DIR),
    "--out", str(SCORED_OUT),
    "--n_jobs", str(NOTEBOOK_N_JOBS),
]
if SKIP_ASPECTS_ON_SCORE:
    score_args.append("--skip_aspects")

run_script("score_and_flag.py", *score_args, *scratch_args())

scored = pd.read_parquet(SCORED_OUT)
cols = [c for c in scored.columns if c in (
    "text", "user_rating", "ai_expected_rating", "delta", "is_anomaly", "ai_hex_color"
)]
scored[cols].head(10)